In [ ]:
import numpy as np
from datetime import datetime
from datetime import timedelta
from stonesoup.types.track import Track
np.random.seed(1600)
from stonesoup.types.detection import Detection
from stonesoup.models.measurement.linear import LinearGaussian
start_time = datetime.now().replace(microsecond=0)
from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, ConstantVelocity
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
q_x = 0.2
q_y = 1
transition_model = CombinedLinearGaussianTransitionModel([ConstantVelocity(q_x),ConstantVelocity(q_y)])
timesteps = [start_time]
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
num_steps = 20
for k in range(1, num_steps + 1):
    timesteps.append(start_time+timedelta(seconds=k))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k]))

measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[0.1, 0],  # Covariance matrix for Gaussian PDF
                          [0, 0.1]])
    )
measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement,
                                  timestamp=state.timestamp,
                                  measurement_model=measurement_model))

from stonesoup.predictor.kalman import AdaptiveKernelKalmanPredictor
predictor = AdaptiveKernelKalmanPredictor(transition_model)
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability
from stonesoup.types.state import ParticleState
from stonesoup.types.array import StateVectors
number_particles = 500
samples = multivariate_normal.rvs(np.array([0, 1,0,1]),
                                  np.diag([0, 0,0,0]),
                                  size=number_particles)
prior = ParticleState(state_vector=StateVectors(samples.T),
                      weight=None,
                      timestamp=start_time)
track = Track()
from stonesoup.updater.kalman import AdaptiveKernelKalmanUpdater
updater = AdaptiveKernelKalmanUpdater()
kernel_weight=None
kernel_covar=None
PSI = None
weight=[]
E_buff = []
Covar_buff = []
Q_buff = []

for measurement in measurements:
    kernel_state_vector, pred_kernel_weight, pred_kernel_covar, prediction = predictor.predict(prior,
                                                                                              PSI= PSI,
                                                                                            kernel_weight=kernel_weight,
                                                                                            kernel_covar=kernel_covar,
                                                                                            timestamp=measurement.timestamp,
                                                                                              samples = samples.T)
    PSI, samples, kernel_weight, kernel_covar, E, Covar, Q  = updater.update(kernel_state_vector = kernel_state_vector, pred_kernel_weight = pred_kernel_weight,
                                                                pred_kernel_covar = pred_kernel_covar, prediction = prediction,
                                                                measurement = measurement, alpha=1, c=1, k=1e-3)
    prior = ParticleState(state_vector=StateVectors(samples.T),
                        weight=None,
                        timestamp=measurement.timestamp)
    E_buff.append(E)
    Covar_buff.append(Covar)
    Q_buff.append(Q)
    weight.append(kernel_weight)
    track.append(prediction)


tracksx_plot = []
truthx_plot = []
truthy_plot = []
tracksy_plot = []
import matplotlib.pyplot as plt
plt.figure()
time = np.arange(num_steps)
for i in time:
  #tracksx_plot.append(np.sum(track[i].state_vector[0]*weightx[i]))
  tracksx_plot.append(track[i].state_vector[0].mean())
  truthx_plot.append(truth[i].state_vector[0])
  truthy_plot.append(truth[i].state_vector[2])
  tracksy_plot.append(track[i].state_vector[2].mean())

plt.figure()
plt.plot(tracksx_plot,tracksy_plot, label="AKKF tracks")
plt.plot(truthx_plot,truthy_plot,label='GT')
plt.legend(loc='upper left')
plt.title("AKKF Tracks")
plt.xlabel("x")
plt.ylabel("y")
plt.savefig("tracks Vs GT")
